In [11]:
# ==========================================
# BLOCO 1: SETUP E IMPORTAÇÕES
# ==========================================
import pandas as pd
import os
import re
import time
import pywhatkit as kit
import pyautogui
from datetime import datetime

# Configurações de exibição do Pandas para facilitar o Debug
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


In [12]:
# ==========================================
# BLOCO 2: GERAÇÃO DE MOCK DATA E ETL
# ==========================================
import pandas as pd
import os

def gerar_6_semestres_mock():
    """
    Gera 6 arquivos CSV independentes simulando a estrutura real de 38 colunas da AEROJR.
    Contém repetições propositais para testar a lógica de deduplicação e limpeza.
    """
    # Cabeçalho oficial identificado nos arquivos originais
    colunas = [
        "Nome", "Diretoria 1º Semestre", "Cargo 1º Semestre", "Semestre de entrada", 
        "Semestre de saída", "E-mail", "Celular", "Telefone fixo", "Nascimento", 
        "Identidade", "Órgão Expedidor", "CPF", "Endereço", "Cidade", "CEP", "Estado", 
        "Banco", "Referência TED", "Agência", "Conta", "Operação", "Pessoa para contato", 
        "Grau de relacionamento", "Celular do contato", "Telefone fixo do contato", 
        "ano atual", "Unnamed: 26", "Chave PIX", "1ª Pessoa para Contato", 
        "Grau de Relacionamento", "Celular do Contato", "Telefone Fixo do Contato", 
        "2ª Pessoa para Contato", "Grau de Relacionamento.1", "Celular do Contato.1", 
        "Telefone Fixo do Contato.1", "Unnamed: 30", "Unnamed: 29"
    ]

    semestres = ['2022_1', '2022_2', '2023_1', '2023_2', '2024_1', '2024_2']
    
    # Base de dados para garantir casos de teste interessantes:
    # 1. Geovana (Projetos): Começa como assessora e termina como diretora.
    # 2. João Silva (Mudança): Marketing em 2022, migra para RH em 2023.
    # 3. Alex José: Fiel ao Marketing, mas muda de cargo.
    membros_teste = [
        ["Geovana Carvalho", "Projetos", "Assessora", "2022_1", "(31) 99408-6737"],
        ["Geovana Carvalho", "Projetos", "Diretora", "2024_2", "31994086737"],
        ["João Silva", "Marketing", "Consultor", "2022_2", "(31) 98888-0001"],
        ["João Silva", "Recursos Humanos", "Assessor", "2023_1", "31 98888-0001"],
        ["Alex José Silva", "Marketing", "Trainee", "2022_1", "(31) 99210-8724"],
        ["Bárbara Luz", "Recursos Humanos", "Assessora", "2022_1", "3199415-7070"]
    ]

    # Garante que a pasta "data" existe
    os.makedirs('data', exist_ok=True)

    for sem in semestres:
        nome_arquivo = f'data/Dados dos Membros - {sem}.csv'
        dados_ano = []
        
        # Gera 6 exemplos por arquivo
        for i in range(6):
            if i < len(membros_teste) and membros_teste[i][3] == sem:
                # Usa membro de teste se o ano coincidir
                m = membros_teste[i]
                linha = [m[0], m[1], m[2], "2021/1", "", f"{m[0].replace(' ', '').lower()}@aerojr.com", m[4]] + [""] * 31
            else:
                # Gera membro genérico para preencher o arquivo
                linha = [f"Membro {i}_{sem}", "Marketing", "Assessor", "2022/1", "", "membro@aerojr.com", "(31) 90000-0000"] + [""] * 31
            
            linha[25] = sem # Coluna 'ano atual'
            dados_ano.append(linha)

        df_temp = pd.DataFrame(dados_ano, columns=colunas)
        df_temp.to_csv(nome_arquivo, index=False)
    
    print(f"✅ 6 arquivos MOCK gerados com sucesso na pasta 'data' (2022_1 a 2024_2).")

def juntar_csvs(pasta, arquivo_saida):
    """Lê múltiplos CSVs de uma pasta e os consolida."""
    dfs = []
    for arquivo in os.listdir(pasta):
        if arquivo.endswith('.csv'):
            caminho_arquivo = os.path.join(pasta, arquivo)
            df = pd.read_csv(caminho_arquivo)
            dfs.append(df)
    
    if dfs:
        df_combinado = pd.concat(dfs, ignore_index=True)
        df_combinado.to_csv(arquivo_saida, index=False)
        print(f"✅ Arquivos combinados com sucesso em: {arquivo_saida}")
        return df_combinado
    return None

# EXECUÇÃO DO FLUXO
pasta_dados = 'data'

# Se os arquivos reais não existirem (cenário GitHub), gera os mocks
# Se a pasta "data" estiver vazia, gera os mocks
if not os.listdir(pasta_dados):
    gerar_6_semestres_mock()
    

# Consolida no df_master
df_master = juntar_csvs(pasta_dados, 'Todos_os_membros_CONSOLIDADO.csv')

# Exibe as primeiras linhas para validação
print(f"\nTotal de linhas no df_master: {len(df_master)}")
df_master.head()

✅ 6 arquivos MOCK gerados com sucesso na pasta 'data' (2022_1 a 2024_2).
✅ Arquivos combinados com sucesso em: Todos_os_membros_CONSOLIDADO.csv

Total de linhas no df_master: 36


,Nome,Diretoria 1º Semestre,Cargo 1º Semestre,Semestre de entrada,Semestre de saída,E-mail,Celular,Telefone fixo,Nascimento,Identidade,Órgão Expedidor,CPF,Endereço,Cidade,CEP,Estado,Banco,Referência TED,Agência,Conta,Operação,Pessoa para contato,Grau de relacionamento,Celular do contato,Telefone fixo do contato,ano atual,Unnamed: 26,Chave PIX,1ª Pessoa para Contato,Grau de Relacionamento,Celular do Contato,Telefone Fixo do Contato,2ª Pessoa para Contato,Grau de Relacionamento.1,Celular do Contato.1,Telefone Fixo do Contato.1,Unnamed: 30,Unnamed: 29
0,Geovana Carvalho,Projetos,Assessora,2021/1,NaN,geovanacarvalho@aerojr.com,(31) 99408-6737,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Membro 1_2022_1,Marketing,Assessor,2022/1,NaN,membro@aerojr.com,(31) 90000-0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Membro 2_2022_1,Marketing,Assessor,2022/1,NaN,membro@aerojr.com,(31) 90000-0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Membro 3_2022_1,Marketing,Assessor,2022/1,NaN,membro@aerojr.com,(31) 90000-0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Alex José Silva,Marketing,Trainee,2021/1,NaN,alexjosésilva@aerojr.com,(31) 99210-8724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# ==========================================
# BLOCO 3: MODELAGEM DE DADOS E REGEX
# ==========================================

class Membro:
    """Abstração de um membro da AEROJR. para manipulação via código."""
    def __init__(self, nome, diretoria, telefone, ano, cargo):
        self.nome = nome
        self.diretoria = diretoria
        self.telefone = self.formatar_telefone(telefone)
        self.ano = ano
        self.cargo = cargo

    def formatar_telefone(self, numero):
        """
        Normaliza números de telefone para o padrão internacional (+55...) 
        utilizando Expressões Regulares (Regex).
        """
        # Remove qualquer caractere que não seja dígito
        numero_limpo = re.sub(r'\D', '', str(numero))
        
        # Remove o '0' inicial, caso exista
        if numero_limpo.startswith("0"):
            numero_limpo = numero_limpo[1:]
        
        # Garante o prefixo do país para o Brasil
        return f"+55{numero_limpo}"

    def __str__(self):
        return f"[{self.ano}] {self.nome} - {self.cargo} ({self.diretoria})"

def instanciar_membros(df):
    """Converte as linhas do DataFrame em uma lista de objetos Membro."""
    lista_membros = []
    for _, row in df.iterrows():
        membro = Membro(
            nome=row['Nome'],
            diretoria=row['Diretoria 1º Semestre'],
            telefone=row['Celular'],
            ano=row['ano atual'],
            cargo=row['Cargo 1º Semestre']
        )
        lista_membros.append(membro)
    return lista_membros

# Execução
membros_objetos = instanciar_membros(df_master)
print(f"📦 {len(membros_objetos)} objetos 'Membro' criados.")

📦 36 objetos 'Membro' criados.


In [14]:
# ==========================================
# BLOCO 4: TRATAMENTO DE DUPLICATAS (PRIORIDADE CRONOLÓGICA)
# ==========================================

def filtrar_contatos_recentes(lista_membros):
    """
    Mantém apenas a entrada mais recente de cada membro,
    documentando o processo de escolha entre duplicatas.
    """
    contatos_unicos = {}
    logs_limpeza = []

    # Ordenação cronológica (mais antigos primeiro) para que
    # os registros novos substituam os velhos no dicionário.
    membros_ordenados = sorted(lista_membros, key=lambda x: x.ano)

    for m in membros_ordenados:
        if m.nome in contatos_unicos:
            m_antigo = contatos_unicos[m.nome]
            logs_limpeza.append(
                f"🔄 ATUALIZAÇÃO: '{m.nome}' encontrado em {m_antigo.ano} ({m_antigo.cargo}) "
                f"-> Atualizado para {m.ano} ({m.cargo})"
            )
        
        # O registro mais recente sempre sobrescreve o anterior.
        contatos_unicos[m.nome] = m

    # Exibição dos casos de limpeza para transparência técnica
    if logs_limpeza:
        print("--- LOG DE DEDUPLICAÇÃO E RESOLUÇÃO DE CONFLITOS ---")
        for log in logs_limpeza:
            print(log)
        print("----------------------------------------------------\n")
    
    return list(contatos_unicos.values())

# Execução
membros_finais = filtrar_contatos_recentes(membros_objetos)
print(f"✅ Processamento concluído: {len(membros_finais)} contatos únicos prontos para a campanha.")

--- LOG DE DEDUPLICAÇÃO E RESOLUÇÃO DE CONFLITOS ---
🔄 ATUALIZAÇÃO: 'João Silva' encontrado em 2022_2 (Consultor) -> Atualizado para 2023_1 (Assessor)
🔄 ATUALIZAÇÃO: 'Geovana Carvalho' encontrado em 2022_1 (Assessora) -> Atualizado para 2024_2 (Diretora)
----------------------------------------------------

✅ Processamento concluído: 34 contatos únicos prontos para a campanha.


In [15]:
# ==========================================
# BLOCO 5: AUTOMAÇÃO DE DISPARO (WHATSAPP)
# ==========================================

def disparar_pesquisa_pos_junior(lista_contatos, diretoria_alvo=None):
    if diretoria_alvo:
        contatos = [m for m in lista_contatos if m.diretoria == diretoria_alvo]
    else:
        contatos = lista_contatos

    for membro in contatos:
        nome = membro.nome.split()[0]
        
        mensagem = f"""Olá, {nome}!

Aqui é a AEROJR., e estamos muito animados para conversar com você novamente! Sabemos que sua passagem pela empresa em {membro.ano} na diretoria de {membro.diretoria} deixou um impacto significativo, e gostaríamos de contar com sua experiência para o futuro da AEROJR.

Estamos em um processo de pesquisa pós-júnior com o objetivo de identificar novas ideias e oportunidades de solução para nossos projetos e clientes. Queremos explorar ideias novas e expandir nossa visão estratégica. Para isso, criamos algumas perguntas orientadoras que ajudarão a conduzir nossa conversa, mas sinta-se à vontade para trazer insights e ideias de maneira natural!

Aqui estão alguns pontos-chave que gostaríamos de abordar:
- Existe alguma ideia de solução que você gostaria de ter visto implementada enquanto esteve conosco?
- Há soluções ou tendências que você vê no seu campo de atuação atual que poderiam ser aplicáveis à AEROJR.?
- Você identifica problemas específicos que novas soluções poderiam resolver para nós ou para nossos clientes?
- Quais oportunidades e benefícios essas soluções trariam para nossa equipe e clientes?
- Pensando estrategicamente, você acha que essas ideias teriam boa aceitação no mercado?

Queremos tanto ideias que possam complementar projetos já existentes quanto sugestões para novos projetos. Além disso, qualquer insight sobre metodologias de execução ou possíveis desafios também será valioso para nossa análise.

Esperamos poder contar com suas ideias e perspectivas para fazer a AEROJR. crescer cada vez mais! Se puder contribuir com qualquer ideia, será incrível!

Grande abraço,  
Equipe AEROJR."""

        try:
            print(f"📤 Enviando para {membro.nome} ({membro.ano})...")
            # Automação de envio
            kit.sendwhatmsg_instantly(membro.telefone, mensagem, wait_time=15)
            time.sleep(2)
            pyautogui.press("enter")
            time.sleep(3)
            pyautogui.hotkey("ctrl", "w")
            print(f"✅ Concluído.")
        except Exception as e:
            print(f"❌ Erro: {e}")

# Para rodar a demonstração:
# disparar_pesquisa_pos_junior(membros_finais, diretoria_alvo='Projetos')